# Boulder recommendation  
## Similarity calculation taking into account ascents and grade 
Ascents: Jaccard similarity  
Grades: Cosine similarity  

## SQLAlchemy session creation

In [26]:
import numpy as np
import pandas as pd


from sqlalchemy.orm import Session
from sqlalchemy import create_engine

DB_URL = "sqlite:///../../matchit.db"

engine = create_engine(DB_URL, echo=False)

session = Session(engine)

In [27]:
import sys

sys.path.append("../../")

## Similarity matrix training **based on similar repetitors**

In [28]:
from sqlalchemy import select
from sqlalchemy.orm import joinedload
from models import boulder
from models.area import Area
from models.ascent import Ascent
from models.boulder import Boulder
from models.crag import Crag
from models.grade import Grade

area_slug = "ticino"

boulders = (
    session.execute(
        select(Boulder)
        .join(Boulder.crag)
        .join(Crag.area)
        .join(Boulder.grade)
        .options(joinedload(Boulder.grade))
        .where(Area.slug == area_slug)
        .order_by(Grade.correspondence, Boulder.name)
    )
    .scalars()
    .all()
)

for index, boulder in enumerate(boulders):
    boulder.similarity_matrix_id = index
    session.add(boulder)
session.commit()

### Database Query

In [29]:
ascents = session.execute(
    select(Ascent.user_id, Boulder.similarity_matrix_id)
    .join(Ascent.boulder)
    .join(Boulder.crag)
    .join(Crag.area)
    .where(Area.slug == area_slug)
).all()
ascents_df = pd.DataFrame(data=ascents, columns=["user_id", "id"])
ascents_df

,user_id,id
0,1,1668
1,2,6555
2,3,0
3,3,809
4,4,810
...,...,...
76937,446,7953
76938,2791,8067
76939,1006,6118
76940,1731,6118


### boulder_user matrix (Pivot table)

In [30]:
boulder_user_matrix = ascents_df.pivot_table(
    index="id",
    columns="user_id",
    aggfunc="size",
    fill_value=0,
    dropna=True,
)
# boulder_user_matrix = boulder_user_matrix[boulder_user_matrix.index < 20]
boulder_ids = boulder_user_matrix.index

In [31]:
display(boulder_user_matrix)

user_id,1,2,3,4,5,6,7,8,9,10,...,4231,4232,4233,4234,4235,4236,4237,4238,4239,4240
id,,,,,,,,,,,,,,,,,,,,,
0,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8170,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8171,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8172,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Conversion to sparse matrix

In [32]:
from scipy.sparse import coo_matrix

boulder_user_matrix = coo_matrix(boulder_user_matrix)
print(boulder_user_matrix)

<COOrdinate sparse matrix of dtype 'int64'
	with 75739 stored elements and shape (6573, 4240)>
  Coords	Values
  (0, 2)	1
  (1, 0)	1
  (2, 7)	1
  (2, 8)	1
  (3, 8)	1
  (3, 11)	1
  (3, 1323)	1
  (4, 15)	1
  (5, 3620)	1
  (6, 15)	1
  (7, 3718)	1
  (8, 23)	1
  (9, 556)	1
  (9, 3720)	1
  (10, 15)	1
  (11, 33)	2
  (11, 34)	1
  (12, 3620)	1
  (13, 15)	1
  (14, 19)	1
  (15, 1264)	1
  (15, 3720)	1
  (15, 3721)	1
  (16, 3620)	1
  (17, 1921)	1
  :	:
  (6564, 418)	1
  (6564, 1392)	1
  (6564, 3186)	1
  (6565, 31)	1
  (6565, 163)	1
  (6565, 378)	1
  (6565, 991)	1
  (6565, 1278)	1
  (6565, 1883)	1
  (6566, 31)	1
  (6566, 399)	1
  (6566, 991)	1
  (6566, 994)	1
  (6566, 1278)	1
  (6567, 31)	1
  (6567, 991)	1
  (6567, 2225)	1
  (6568, 31)	1
  (6568, 1204)	1
  (6568, 2225)	1
  (6569, 1278)	1
  (6570, 1278)	1
  (6571, 418)	1
  (6571, 1392)	1
  (6572, 378)	1


### Ascents similarity training

In [33]:
def jaccard_pairwise_similarity(X):
    """Compute the Jaccard similarity between pairs of boulders based on user ascents.
    Args:
        X (scipy.sparse.csr_matrix): Sparse matrix of shape (n_boulders, n_users)
            where each entry indicates whether a user has ascended a boulder.
    Returns:
        scipy.sparse.coo_matrix: Sparse matrix of shape (n_boulders, n_boulders)
            containing the Jaccard similarity between each pair of boulders.
    """
    # CSR matrix storing the number of shared ascents for each pair of
    # boulders sharing at least one ascent

    intersection = X @ X.T

    # 1D array storing the total number of ascent for each boulder
    row_sums = np.asarray(X.sum(axis=1)).ravel()

    # intersection decomposition for calculation on 1D arrays
    rows, cols = intersection.nonzero()
    intersection_data = intersection.data

    union = row_sums[rows] + row_sums[cols] - intersection_data

    # Avoid division by zero
    mask = union > 0
    jaccard = np.zeros_like(intersection_data, dtype=np.float32)
    jaccard[mask] = intersection_data[mask] / union[mask]

    # Index remapping based on the boulder ids
    new_rows = boulder_ids[rows]
    new_cols = boulder_ids[cols]

    return coo_matrix(
        (jaccard, (new_rows, new_cols)),
        dtype=np.float32,
    )

similarity_ascents = jaccard_pairwise_similarity(boulder_user_matrix)

sparsity = 1.0 - similarity_ascents.nnz / (
    similarity_ascents.shape[0] * similarity_ascents.shape[1]
)

print(f"Sparsity: {sparsity:.2%}")

Sparsity: 96.10%


In [34]:
# similarity_ascents_df = pd.DataFrame(similarity_ascents.toarray())
# display(similarity_ascents_df)

## Similarity matrix training **based on similar features**

### Database query and dataframe creation

In [35]:
# Data extraction
boulders_list = [
    {
        "id": boulder.similarity_matrix_id,
        "grade": boulder.grade.correspondence,
    }
    for boulder in boulders
]

# Dataframe setup
boulders_df = pd.DataFrame(boulders_list)
# boulders_df = boulders_df[boulders_df.id < 20]
boulder_ids = boulders_df.id
boulders_df.head()


,id,grade
0,0,12
1,1,12
2,2,12
3,3,12
4,4,12


### Grade

#### Fuzzy one-hot grade vector

In [36]:
max_grade = boulders_df.grade.max()
min_grade = boulders_df.grade.min()
grade_df = pd.get_dummies(boulders_df.grade, dtype=np.float32)
grade_df

,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8170,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8171,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8172,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8173,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [37]:
def grade_update(row, min_grade, max_grade):
    grade_index = row.idxmax()

    if grade_index == 0:
        return row

    values = np.array([0.5, 0.75, 0.75, 0.5], dtype=np.float32)
    offsets = np.array([-2, -1, 1, 2])

    for offset, value in zip(offsets, values):
        current_column = grade_index + offset
        if min_grade < current_column <= max_grade:
            row[current_column] = value
    return row


grade_df = grade_df.apply(
    lambda row: grade_update(row, min_grade=min_grade, max_grade=max_grade),
    axis=1,
)
grade_df.fillna(0, inplace=True)
display(grade_df)

,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30
0,1.0,0.75,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00
1,1.0,0.75,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00
2,1.0,0.75,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00
3,1.0,0.75,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00
4,1.0,0.75,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8170,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.75,1.00,0.75
8171,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.75,1.00,0.75
8172,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.75,1.00,0.75
8173,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.75,1.00,0.75


#### Conversion to sparse matrix

In [38]:
grade = coo_matrix(grade_df)

#### Grade similarity training

In [39]:
from time import perf_counter
from sklearn.metrics.pairwise import cosine_similarity

# Cosine training
start = perf_counter()
similarity_grade = cosine_similarity(grade)
end = perf_counter()
print(f"Cosine calculation time: {end - start:.4f}")


# Re-indexing to match database
start = perf_counter()
similarity_grade = coo_matrix(similarity_grade)
end = perf_counter()
print(f"COO conversion time: {end - start:.4f}")

# Sparcity
sparsity = 1 - similarity_grade.nnz / (
    similarity_grade.shape[0] * similarity_grade.shape[1]
)
print(f"Sparcity: {sparsity:.2f}")

# similarity_grade_df = pd.DataFrame(similarity_grade.toarray())
# display(similarity_grade_df)

Cosine calculation time: 0.1366
COO conversion time: 0.9363
Sparcity: 0.44


In [40]:
print(similarity_grade)

<COOrdinate sparse matrix of dtype 'float32'
	with 37205657 stored elements and shape (8175, 8175)>
  Coords	Values
  (0, 0)	1.0
  (0, 1)	1.0
  (0, 2)	1.0
  (0, 3)	1.0
  (0, 4)	1.0
  (0, 5)	1.0
  (0, 6)	1.0
  (0, 7)	1.0
  (0, 8)	1.0
  (0, 9)	1.0
  (0, 10)	1.0
  (0, 11)	1.0
  (0, 12)	1.0
  (0, 13)	1.0
  (0, 14)	1.0
  (0, 15)	1.0
  (0, 16)	1.0
  (0, 17)	1.0
  (0, 18)	1.0
  (0, 19)	1.0
  (0, 20)	1.0
  (0, 21)	1.0
  (0, 22)	1.0
  (0, 23)	1.0
  (0, 24)	1.0
  :	:
  (8174, 8150)	0.7163352966308594
  (8174, 8151)	0.7163352966308594
  (8174, 8152)	0.7163352966308594
  (8174, 8153)	0.7163352966308594
  (8174, 8154)	0.7163352966308594
  (8174, 8155)	0.7163352966308594
  (8174, 8156)	0.7163352966308594
  (8174, 8157)	0.7163352966308594
  (8174, 8158)	0.7163352966308594
  (8174, 8159)	0.7163352966308594
  (8174, 8160)	0.7163352966308594
  (8174, 8161)	0.7163352966308594
  (8174, 8162)	0.7163352966308594
  (8174, 8163)	0.7163352966308594
  (8174, 8164)	0.7163352966308594
  (8174, 8165)	0.90371280908

## Coherence check

In [41]:
print(type(similarity_ascents))
print(type(similarity_grade))

print(similarity_ascents.shape)
print(similarity_grade.shape)

print(similarity_ascents.nnz)
print(similarity_grade.nnz)

print(similarity_ascents.dtype)
print(similarity_grade.dtype)

<class 'scipy.sparse._coo.coo_matrix'>
<class 'scipy.sparse._coo.coo_matrix'>
(8175, 8175)
(8175, 8175)
2603337
37205657
float32
float32


## Data cleaning
Removing all values in similarity_grade < 0.5 (grade difference >= 2)  
Removing of all non zero values in similarity_ascents and similarity_style where similarity_grade == 0

In [42]:
similarity_grade_cleaned = similarity_grade.copy()
similarity_grade_cleaned.data[similarity_grade_cleaned.data < 0.5] = 0
similarity_grade_cleaned.eliminate_zeros()

In [43]:
similarity_grade_cleaned = similarity_grade_cleaned.tocsr()

In [44]:
from scipy.sparse import csr_matrix

def matrix_cleaning(cleaning_matrix: csr_matrix, matrix_to_clean: coo_matrix):
    """Remove all values in matrix_to_clean that are not indexed in cleaning_matrix
    
    :parameters:
    cleaning_matrix: CSR matrix that serves as reference for data existence
    matrix_to_clean: COO matrix from which some indexes are removed"""
    
    # Boolean mask creation (1D array) - Fancy indexing converted to boolean
    # Check if cleaning_matrix contains the indexes of matrix_to_clean
    mask = cleaning_matrix[matrix_to_clean.row, matrix_to_clean.col].A1 != 0

    # Fancy indexing to remove values from matrix_to_clean that are equal to 0 
    # in cleaning_matrix
    new_rows = matrix_to_clean.row[mask]
    new_cols = matrix_to_clean.col[mask]
    new_data = matrix_to_clean.data[mask]

    return csr_matrix(
        (new_data, (new_rows, new_cols)), shape=matrix_to_clean.shape
    )

similarity_ascents_cleaned = matrix_cleaning(similarity_grade_cleaned, similarity_ascents)

In [45]:
print(
    1
    - similarity_ascents.nnz
    / (similarity_ascents.shape[0] * similarity_ascents.shape[1])
)
print(
    1
    - similarity_ascents_cleaned.nnz
    / (
        similarity_ascents_cleaned.shape[0]
        * similarity_ascents_cleaned.shape[1]
    )
)
print(
    1
    - similarity_grade.nnz
    / (similarity_grade.shape[0] * similarity_grade.shape[1])
)
print(
    1
    - similarity_grade_cleaned.nnz
    / (similarity_grade_cleaned.shape[0] * similarity_grade_cleaned.shape[1])
)

0.9610457481132341
0.9797167092182664
0.4432843176313255
0.6580589363035285


## Matrix saving

In [46]:
from scipy.sparse import save_npz

# save_npz("../../similarity_ascent.npz", similarity_ascents_cleaned)
# save_npz("../../similarity_grade.npz", similarity_grade_cleaned)

## Recommendation example


In [47]:
def recommend_boulders(input_boulders, top_n=5, alpha=0.7, beta=0.3):

    ascents = similarity_ascents_cleaned[:, input_boulders].sum(axis=1).A1
    grade = similarity_grade_cleaned[:, input_boulders].sum(axis=1).A1

    ascents[input_boulders] = 0
    grade[input_boulders] = 0

    sim_scores = alpha * ascents + beta * grade

    best_boulders = np.argsort(-sim_scores)[:top_n]

    return best_boulders.tolist()


recommendations = recommend_boulders([8112], top_n=20)


boulders = session.execute(
    select(Boulder.name, Boulder.similarity_matrix_id).filter(
        Boulder.similarity_matrix_id.in_(recommendations)
    )
).all()

boulder_dict = {
    boulder_id: boulder_name for boulder_name, boulder_id in boulders
}
boulder_names = [(boulder_dict[boulder_id], boulder_id) for boulder_id in recommendations]
display(boulder_names)

[('Big Kat', 8089),
 ('Primitivo Stand', 8040),
 ('Casavino', 8092),
 ('Heritage', 8104),
 ('Tomba', 8126),
 ('Momentum', 8111),
 ('The Story Of 2 Worlds', 8159),
 ('Off the Wagon', 8114),
 ('Mithril', 8033),
 ('Insanity of Grandeur', 8105),
 ('Fight Club', 8097),
 ('Ninjutsu ', 8113),
 ('Bella Luna', 7978),
 ('Kings of Sonlerto', 8017),
 ('Confessions', 7990),
 ('Zingaro', 8070),
 ('Santoku', 8043),
 ('Der mit dem Fels tanzt', 8140),
 ('Primitivo', 8154),
 ('Dreamtime', 8142)]

In [48]:
from sqlalchemy import func


query = session.execute(
    select(
        Boulder.id,
        Boulder.name,
        func.count(Ascent.user_id),
        Boulder.similarity_matrix_id,
    )
    .join(Boulder.ascents)
    .where(Boulder.name_normalized.like("%santoku%"))
    .group_by(Boulder.id)
).all()

for boulder_id, boulder_name, ascents_count, similarity_matrix_id in query:
    print(
        f"boulder {boulder_id} ({boulder_name}), ascents {ascents_count}, similarity_matrix_id {similarity_matrix_id}"
    )

boulder 7249 (Santoku), ascents 34, similarity_matrix_id 8043
boulder 7250 (Santoku stand), ascents 13, similarity_matrix_id 7649


In [52]:
name = "silvretta"

total_boulders = session.execute(
    select(func.count(Boulder.id))
    .join(Boulder.crag)
    .join(Crag.area)
    .where(Area.name_normalized == name)
).scalar()

boulders_with_ascents = session.execute(
    select(func.count(func.distinct(Boulder.id)))
    .join(Boulder.crag)
    .join(Boulder.ascents)
    .join(Crag.area)
    .where(Area.name_normalized == name)
).scalar()

print(f"Boulders with ascents: {boulders_with_ascents}/{total_boulders}")
print(f"Percentage: {boulders_with_ascents/total_boulders:.2%}")
print(f"Number of duplicates: {total_boulders - boulders_with_ascents}")

Boulders with ascents: 500/648
Percentage: 77.16%
Number of duplicates: 148
